# 📘 Bài Giảng Thực Hành: Hồi Quy Tuyến Tính Đa Biến (Multiple Linear Regression)
**Mục tiêu bài giảng:**
1. Hiểu và áp dụng phương trình Hồi quy Tuyến tính Đa biến: $y = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_n x_n$.
2. Sử dụng `LinearRegression` từ Scikit-Learn để tìm các hệ số trọng số tối ưu.
3. Đánh giá chất lượng mô hình với các chỉ số định lượng: $R^2$ Score và RMSE.
4. Áp dụng thuật toán **Loại Bỏ Ngược (Backward Elimination)** bằng kiểm định p-value qua `statsmodels.api.OLS` để chọn lọc biến có ý nghĩa thống kê.
5. Trực quan hóa giá trị Thực tế vs Dự đoán và mức độ tác động của các biến đặc trưng.

---
### 📊 Tập dữ liệu thực hành: `50_Startups.csv`
- **Features ($X$)**:
  - `R&D Spend`: Chi phí Nghiên cứu & Phát triển ($).
  - `Administration`: Chi phí Quản trị & Vận hành ($).
  - `Marketing Spend`: Chi phí Tiếp thị & Quảng cáo ($).
- **Target ($y$)**: `Profit` - Lợi nhuận của doanh nghiệp Startup ($).


## 1. Đọc & Khám phá Dữ liệu


In [ ]:
import pandas as pd

dataset = pd.read_csv('50_Startups.csv')
print('Số lượng mẫu:', len(dataset))
dataset.head()


📌 **Expected Output:**
```
Số lượng mẫu: 50
   R&D Spend  Administration  Marketing Spend       State     Profit
0  165349.20       136897.80        471784.10    New York  192261.83
1  162597.70       151377.59        443898.53  California  191792.06
2  153441.51       101145.55        407934.54     Florida  191050.39
3  144372.41       118671.85        383199.62    New York  182901.99
4  142107.34        91391.77        366168.42     Florida  166187.94
```


## 2. Chuẩn bị Đặc trưng & Phân chia Tập Huấn luyện / Kiểm thử (Train / Test)
Ta chỉ trích xuất các biến độc lập dạng số (`R&D Spend`, `Administration`, `Marketing Spend`) để dự đoán lợi nhuận (`Profit`).


In [ ]:
from sklearn.model_selection import train_test_split

feature_names = ['R&D Spend', 'Administration', 'Marketing Spend']
X = dataset[feature_names].values
y = dataset['Profit'].values

# Chia tập Train/Test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print(f'Số lượng mẫu Train: {len(X_train)} | Số lượng mẫu Test: {len(X_test)}')


📌 **Expected Output:**
```
Số lượng mẫu Train: 40 | Số lượng mẫu Test: 10
```


## 3. Huấn luyện Mô hình Hồi quy Tuyến tính Đa biến
Mô hình ước lượng các trọng số $w_0, w_1, w_2, w_3$ cho phương trình:
$$\text{Profit} = w_0 + w_1 \cdot \text{R\&D} + w_2 \cdot \text{Admin} + w_3 \cdot \text{Marketing}$$


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error

regressor = LinearRegression()
regressor.fit(X_train, y_train)
y_pred = regressor.predict(X_test)

print(f'Hệ số chặn (Intercept w0): {regressor.intercept_:,.2f}')
for name, coef in zip(feature_names, regressor.coef_):
    print(f'Trọng số {name} (w): {coef:.4f}')

print('-' * 40)
print(f'R² Score trên Test Set: {r2_score(y_test, y_pred):.4f}')
print(f'RMSE trên Test Set:     ${root_mean_squared_error(y_test, y_pred):,.2f}')


📌 **Expected Output:**
```
Hệ số chặn (Intercept w0): 42,989.01
Trọng số R&D Spend (w): 0.7788
Trọng số Administration (w): 0.0294
Trọng số Marketing Spend (w): 0.0347
----------------------------------------
R² Score trên Test Set: 0.9394
RMSE trên Test Set:     $8,803.78
```


## 4. Thuật toán Loại Bỏ Ngược (Backward Elimination) với Statsmodels OLS
Đánh giá mức độ ý nghĩa thống kê của từng biến dựa trên giá trị **\text{-value}$** (ngưỡng $\alpha = 0.05$).


In [9]:
import numpy as np
import statsmodels.api as sm

# Thêm cột hệ số tự do w0 (toàn số 1: x0 = 1) vào ma trận X
X_opt = np.append(arr=np.ones((X.shape[0], 1)).astype(float), values=X, axis=1)
all_feature_names = ['Intercept (w0)'] + feature_names

# Huấn luyện mô hình OLS và truyền xname để hiển thị rõ tên từng đặc trưng
regressor_ols = sm.OLS(endog=y, exog=X_opt).fit()
print(regressor_ols.summary(xname=all_feature_names))


                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.951
Model:                            OLS   Adj. R-squared:                  0.948
Method:                 Least Squares   F-statistic:                     296.0
Date:                Sat, 26 Sep 2026   Prob (F-statistic):           4.53e-30
Time:                        10:49:40   Log-Likelihood:                -525.39
No. Observations:                  50   AIC:                             1059.
Df Residuals:                      46   BIC:                             1066.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept (w0)   5.012e+04   6572.353     

📌 **Expected Output:**
```
Kết quả OLS Regression Results hiển thị:
- R&D Spend có p-value = 0.000 (< 0.05): Tác động cực kỳ mạnh và có ý nghĩa thống kê tới Lợi nhuận.
- Administration (p = 0.608) và Marketing Spend (p = 0.123): Có p-value > 0.05.
```


## 5. Trực quan hóa So sánh Thực tế vs Dự đoán & Trọng số Mô hình


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Đồ thị 1: Actual vs Predicted
axes[0].scatter(y_test, y_pred, color='blue', s=80, alpha=0.7)
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=2)
axes[0].set_title('Thực tế vs Dự đoán (Actual vs Predicted)', fontweight='bold')
axes[0].set_xlabel('Lợi nhuận Thực tế ($)')
axes[0].set_ylabel('Lợi nhuận Dự đoán ($)')

# Đồ thị 2: Trọng số của các đặc trưng
colors = ['green' if c > 0 else 'red' for c in regressor.coef_]
axes[1].barh(feature_names, regressor.coef_, color=colors, alpha=0.8)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Trọng số Hệ số của Đặc trưng (Feature Coefficients)', fontweight='bold')
axes[1].set_xlabel('Hệ số w_i')

plt.tight_layout()
plt.show()


📌 **Expected Output:**
- Đồ thị bên trái: Các điểm dự đoán bám rất sát đường chéo lý tưởng $y = x$ ($R^2 \approx 94\%$).
- Đồ thị bên phải: `R&D Spend` có trọng số dương vượt trội ($w \approx 0.78$), là động lực chính thúc đẩy lợi nhuận doanh nghiệp.

---
### 💡 Câu hỏi Thảo luận cho Học viên:
1. Ý nghĩa kinh tế của hệ số $w_1 = 0.7788$ của `R&D Spend` là gì? (Gợi ý: Doanh nghiệp chi thêm $1 vào R&D thì lợi nhuận tăng bao nhiêu?)
2. Dựa vào kết quả kiểm định $p\text{-value}$ từ OLS, nếu phải lược bỏ bớt một biến để tinh giản mô hình, bạn sẽ chọn biến nào trước tiên?
